- PDF 랜더링 :PyMuPDF(fitz) 사용해서 pdf페이지를 PNG 이미지로 변환 및 저장
- 비전 기반 페이지 요약 : gpt-5.4-nano  각 페이지를 해석해서 표 구조를 포함한 텍스트 요약문을 도출
- 요약-이미지 인덱싱 : 요약문 텍스트는 chromadb 임베딩저장 메타데이터에 원본 이미지 경로를 매핑
- 검색 및 리트리버 : 질문에 맞는 페이지 요약문이 탐지되면 관련 원본 페이지 이미지를 확보
- 멀티모달 답변 생성 : gpt-4.5-nano 질문과 원본페이지 이미지를 동시에 llm에 제공해서 시각적인 데이터(표구조포함)를 직접 해석한 답변을 완성
- 웹검색 Fallback연동 : 내부지식이 부족한 경우 네이버openai/duckduckgo  실시간 웹 검색으로 답변을 보강

### STEP1 환경설정

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')[:20]

'sk-proj-FRucyNlpe5gd'

### STEP2  PyMuPDF(fitz)를 이용한 PDF의 고화질 이미지 변환

In [ ]:
import fitz
import glob
os.makedirs('./docs_images', exist_ok=True)
pdf_files = glob.glob("./doc/*.pdf")
print('pdf 목록 :',pdf_files)
page_images = []
for pdf in pdf_files:
    print(f'이미지 랜더링 중..{pdf}')
    doc = fitz.open(pdf)    
    base_name = os.path.splitext(os.path.basename(pdf))[0]
    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)
        pix = page.get_pixmap(dpi=150)
        image_path = f'./docs_images/page_{base_name}_{page_idx}.png'
        pix.save(image_path)
        page_images.append({
            "image_path":image_path,
            "source":pdf,
            "page":page_idx
        })
print(f"총 {len(page_images)}개의 pdf 페이지 이미지 렌더링 완료")
    

pdf 목록 : ['./doc\\01._북한_원산갈마해안관광특별구법에_대한_법적_고찰.pdf', './doc\\02._인도네시아_헌법의_특성과_체계.pdf', './doc\\03._미등록_아동_방지를_위한_출생등록될_권리의_법제화_방안.pdf', './doc\\법제시론_국회_입법기능의_정상화_전학선.pdf']
이미지 랜더링 중.../doc\01._북한_원산갈마해안관광특별구법에_대한_법적_고찰.pdf
('01._북한_원산갈마해안관광특별구법에_대한_법적_고찰', '.pdf')
